# Baseline Benchmark

Runs the initial LLM call on all benchmark queries, evaluates the results, and saves everything to disk (including `prompt.txt` per query so a refined run can load it later).

In [1]:
import importlib
import llm as llm_module
import filter as filter_module
importlib.reload(llm_module)
importlib.reload(filter_module)
from llm import call_llm, build_prompt
from filter import filter_services

import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from analysis import analyze
import output as output_module

## Configuration

In [2]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

FILTER_SERVICES = True  # BM25 endpoint pre-filtering
TOP_K = 5               # endpoints to keep per service when FILTER_SERVICES=True

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Workers: {MAX_WORKERS} | Model: {MODEL} | Filter: {FILTER_SERVICES} (top_k={TOP_K})")

Loaded 11 benchmark sets (total available: 11)
Total queries: 50 (of 110 available)
Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro | Filter: True (top_k=5)


## Prompt Template

In [3]:
PROMPT_TEMPLATE = '''You are an expert software engineer performing REST service composition.

You are given:


A natural-language task description.
One or more REST API specifications (OpenAPI/Swagger).


Your job is to write a single self-contained Python script that fulfills the task by calling the necessary endpoints, in the correct order, passing data from earlier responses into later requests as required.

Output contract


Respond with raw Python source code ONLY. Your entire response must be directly executable by a Python interpreter with no edits.
The response must begin with an import statement (e.g. import requests). Do not emit markdown, code fences (```), backticks, language tags, comments, docstrings, prose, or trailing notes — nothing but code.
Import requests and define exactly one function named compose. Place all request logic inside compose. Do NOT call compose.
compose must return the final result that answers the task.


Composition rules


Use ONLY endpoints, HTTP methods, paths, parameters, and fields defined in the provided specifications. Do not invent endpoints, parameters, response fields, or hosts.
Build each request URL by joining the base URL from the spec\'s servers field with the operation path. If several servers are listed, use the first.
Place parameters exactly as the spec defines them: substitute path parameters into the URL, pass query parameters via params=, headers via headers=, and request bodies via json= (or data= for form bodies). Use the exact parameter and field names from the spec.
If the spec defines a security scheme (API key, bearer token, etc.), include it where the spec requires it (header or query). When no concrete value is supplied in the task, use a clearly named placeholder constant (e.g. API_KEY = "<API_KEY>").
Parse JSON responses with .json(), extract the specific fields you need, and feed them into subsequent calls. Chain calls so each step\'s output drives the next.
Iterate when the task requires processing a collection; otherwise issue each required call once.
Call only the endpoints strictly required to satisfy the task. Do not call supplementary endpoints whose output is not used as input to a later step or as the final result. When two endpoints seem relevant to the same task step, pick the one whose description most directly matches — do not call both.


Code-quality rules (the output is statically analyzed)


Use only the requests library and the Python standard library. No other third-party imports.
Every name must be defined before use. No undefined references, no unused imports or variables, no placeholders like ... or TODO.
Add type annotations to the compose signature and its return type, and to local variables where the type is clear. Import every typing symbol you reference (from typing import Any). Annotate decoded JSON as Any or dict[str, Any] rather than guessing concrete shapes.
Write valid, parseable Python with explicit, deterministic control flow.


Output shape (format example only — do NOT copy its logic or endpoints)

import requests
from typing import Any

def compose() -> Any:
    base_url = "https://api.example.com"
    first = requests.get(f"{{base_url}}/items", params={{"limit": 1}}).json()
    item_id = first["data"][0]["id"]
    detail = requests.get(f"{{base_url}}/items/{{item_id}}").json()
    return detail

Task

{query}

Source

{services_block}
'''


## Initial LLM Call

In [4]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

tasks = []
for benchmark in benchmark_sets:
    if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
        break
    for query_index, query in enumerate(benchmark['queries'], start=1):
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        tasks.append((benchmark, query_index, query))

def _call_initial(args):
    benchmark, query_index, query = args
    services = (
        filter_services(benchmark['services'], query['query'], top_k=TOP_K)
        if FILTER_SERVICES else benchmark['services']
    )
    prompt = build_prompt(services, query['query'], PROMPT_TEMPLATE)
    t0 = time.time()
    generated, usage = call_llm(prompt, MODEL, '')
    elapsed = time.time() - t0
    generated += '\n\ncompose()'
    return {
        'query_index': query_index,
        'sector_name': benchmark['name'],
        'query': query,
        'prompt': prompt,
        'generated': generated,
        'service_files': benchmark.get('service_files', []),
        'model': MODEL,
        '_elapsed': elapsed,
        '_usage': usage,
    }

sector_results = []
call_times = []
run_start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_initial, t): t for t in tasks}
    for future in as_completed(futures):
        result = future.result()
        sector_results.append(result)
        call_times.append(result['_elapsed'])

        done = len(sector_results)
        avg_t = sum(call_times) / len(call_times)
        remaining = total_queries - done
        eta_s = remaining * avg_t / min(remaining, MAX_WORKERS) if remaining else 0
        usage = result['_usage']
        tok_str = (f"prompt={usage['prompt_tokens']} comp={usage['completion_tokens']}"
                   if usage else "tokens=n/a")
        print(
            f"[{done:>3}/{total_queries}] [{result['sector_name']}] Q{result['query_index']}"
            f" | {result['_elapsed']:.1f}s | {tok_str}"
            f" | avg {avg_t:.1f}s | ETA ~{eta_s:.0f}s"
        )

sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))
print(f"\nDone. Total wall time: {time.time() - run_start:.1f}s")

[  1/50] [01-energy] Q8 | 9.8s | prompt=5452 comp=356 | avg 9.8s | ETA ~48s
[  2/50] [01-energy] Q4 | 11.8s | prompt=5581 comp=448 | avg 10.8s | ETA ~52s
[  3/50] [01-energy] Q5 | 11.8s | prompt=5370 comp=492 | avg 11.1s | ETA ~52s
[  4/50] [01-energy] Q1 | 12.0s | prompt=5483 comp=522 | avg 11.3s | ETA ~52s
[  5/50] [01-energy] Q2 | 14.1s | prompt=5512 comp=584 | avg 11.9s | ETA ~53s
[  6/50] [01-energy] Q7 | 14.3s | prompt=5544 comp=615 | avg 12.3s | ETA ~54s
[  7/50] [01-energy] Q6 | 15.9s | prompt=5614 comp=658 | avg 12.8s | ETA ~55s
[  8/50] [01-energy] Q9 | 16.3s | prompt=5439 comp=688 | avg 13.2s | ETA ~56s
[  9/50] [01-energy] Q10 | 19.2s | prompt=5476 comp=788 | avg 13.9s | ETA ~57s
[ 10/50] [02-materials] Q6 | 4.9s | prompt=5619 comp=248 | avg 13.0s | ETA ~52s
[ 11/50] [01-energy] Q3 | 25.5s | prompt=5466 comp=1227 | avg 14.1s | ETA ~55s
[ 12/50] [02-materials] Q10 | 7.8s | prompt=5517 comp=494 | avg 13.6s | ETA ~52s
[ 13/50] [02-materials] Q7 | 13.4s | prompt=5421 comp=755 |

## Evaluate

In [5]:
for result in sector_results:
    initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
    result['initial_metrics'] = initial_metrics
    print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")

Initial evaluation - Query 1
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   []
  Extra:     []
Initial evaluation - Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Initial evaluation - Query 3
  Precision: 0.67
  Recall:    0.75
  F1:        0.71
  Extracted: ['GET /energy-patterns', 'GET /equipment-monitoring', 'GET /real-time-data', 'GET /reports/m

## Save Outputs

In [6]:
from datetime import datetime
importlib.reload(output_module)
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + "_baseline"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': True,
    'model': MODEL,
    'filter_services': FILTER_SERVICES,
    'top_k': TOP_K if FILTER_SERVICES else None,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-28_17-32-41_baseline


## Summary

In [7]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.74
  Average Recall:    0.82
  Average F1:        0.77
  Avg. Missing Endpoints: 1.72
  Avg. Extra Endpoints:   2.34
  Correct Compositions: 4/50 (8.0%)
